# 05 · Compare models + finalize hypothesis test

Pulls everything pushed by notebooks 03 + 04 from the HF model repo, builds the
comparison table the study requires (Accuracy / Precision / Recall / F1 across
RF, IF, and the rule-based baseline), renders ROC curves, and writes a final
**auto-generated model card** to the repo so HF presents the metrics on the repo page.

In [ ]:
import sys, subprocess
for p in ('huggingface_hub','pandas','matplotlib','plotly'):
    try: __import__(p)
    except ImportError: subprocess.check_call([sys.executable,'-m','pip','install','-q',p])
print('deps OK')

In [ ]:
import os, json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from huggingface_hub import HfApi, hf_hub_download, whoami

HF_TOKEN = os.environ['HF_TOKEN']
HF_USERNAME = os.environ.get('HF_USERNAME') or whoami(token=HF_TOKEN)['name']
MODEL_REPO = f'{HF_USERNAME}/fraud-detection-models'
metadata = json.loads(Path(hf_hub_download(repo_id=MODEL_REPO, filename='metadata.json', token=HF_TOKEN)).read_text())
print('Loaded metadata keys:', list(metadata.keys()))

## 1. Metrics table

In [ ]:
rows = []
if 'paysim' in metadata:
    rows.append({'model': 'RandomForest (PaySim)',   **{k: metadata['paysim']['rf'].get(k)          for k in ('accuracy','precision','recall','f1','roc_auc')}})
    rows.append({'model': 'Rule-based  (PaySim)',   **{k: metadata['paysim']['rule_based'].get(k)  for k in ('accuracy','precision','recall','f1','roc_auc')}})
if 'custom' in metadata:
    rows.append({'model': 'IsolationForest (Custom)',**{k: metadata['custom']['isolation_forest'].get(k) for k in ('accuracy','precision','recall','f1','roc_auc')}})
    rows.append({'model': 'Rule-based  (Custom)',   **{k: metadata['custom']['rule_based'].get(k)  for k in ('accuracy','precision','recall','f1','roc_auc')}})
table = pd.DataFrame(rows).set_index('model')
table

## 2. ROC curves (RF vs IF)

In [ ]:
plt.figure(figsize=(8, 6))
for src in ('paysim','custom'):
    if src not in metadata: continue
    block = metadata[src]['rf'] if src == 'paysim' else metadata[src]['isolation_forest']
    roc = block.get('roc_curve')
    if roc:
        plt.plot(roc['fpr'], roc['tpr'], label=f"{block['label']} (AUC={roc['auc']:.3f})")
plt.plot([0,1],[0,1], '--', color='gray', label='random')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC curves — ML models vs random'); plt.legend(); plt.grid(alpha=0.3)
plt.show()

## 3. Hypothesis test summary

In [ ]:
for src in ('paysim','custom'):
    block = metadata.get(src, {}).get('mcnemar')
    if not block: continue
    print(f'\n=== {src.upper()} ===')
    print(f"p-value      : {block['p_value']:.6f}")
    print(f"reject H1?   : {block['reject_null_at_0.05']}")
    print(f"interpretation: {block['interpretation']}")
    print(f"contingency  : {block['table']}")

## 4. Generate and upload the model card README

In [ ]:
rf = metadata.get('paysim', {}).get('rf', {})
iso = metadata.get('custom', {}).get('isolation_forest', {})
card = f'''---
license: mit
tags:
  - fraud-detection
  - random-forest
  - isolation-forest
  - tabular-classification
  - banking
---

# Fraud Detection Models — Random Forest + Isolation Forest

Trained as part of an ML-FDS study for commercial banks and credit unions in Cameroon.

## Artifacts

| File | Purpose |
|---|---|
| `random_forest.joblib` | RandomForestClassifier on PaySim (SMOTE-balanced) |
| `isolation_forest.joblib` | IsolationForest on the custom dataset (legit-only training) |
| `paysim_preprocessor.joblib` | Fitted PaySimPreprocessor |
| `custom_preprocessor.joblib` | Fitted CustomPreprocessor |
| `metadata.json` | Full metrics, feature importances, hypothesis test |

## Headline metrics

### Random Forest (PaySim, supervised)
- Accuracy: **{rf.get('accuracy','n/a')}**
- Precision: **{rf.get('precision','n/a')}**
- Recall: **{rf.get('recall','n/a')}**
- F1: **{rf.get('f1','n/a')}**
- ROC-AUC: **{rf.get('roc_auc','n/a')}**

### Isolation Forest (Custom, unsupervised)
- Accuracy: **{iso.get('accuracy','n/a')}**
- Precision: **{iso.get('precision','n/a')}**
- Recall: **{iso.get('recall','n/a')}**
- F1: **{iso.get('f1','n/a')}**

## Usage

```python
from huggingface_hub import hf_hub_download
import joblib

rf_path = hf_hub_download(repo_id="{MODEL_REPO}", filename="random_forest.joblib")
pre_path = hf_hub_download(repo_id="{MODEL_REPO}", filename="paysim_preprocessor.joblib")
rf = joblib.load(rf_path)
preprocessor = joblib.load(pre_path)
```

## Class imbalance handling
PaySim fraud rate is ~0.13%. We apply SMOTE on the training fold only, plus `class_weight=balanced`.

## Hypothesis test (H1 vs HA)
See `metadata.json -> paysim.mcnemar` and `metadata.json -> custom.mcnemar`.
'''

Path('/tmp/model_card.md').write_text(card)
HfApi().upload_file(path_or_fileobj='/tmp/model_card.md', path_in_repo='README.md',
                    repo_id=MODEL_REPO, repo_type='model', token=HF_TOKEN,
                    commit_message='Update model card')
print('Model card pushed.')